# Explore movie data

This notebook reads the local cache and never fetches automatically. Run `uv run streamvault` first if needed. Reference frames contain each entity once; bridge and credit frames preserve movie relationships by ID. Person popularity stays on credit rows because it varies between cached movie responses.


In [1]:
from pathlib import Path

import pandas as pd

from streamvault.cache import load_cached_records

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
cache_path = project_root / "data" / "raw" / "movies.jsonl"
records = list(load_cached_records(cache_path).values())


In [2]:
if records:
    records[0].keys()
    print(f"{len(records)} movies were retrieved")
else:
    print("No cached movie records. Run `uv run streamvault` first.")

112 movies were retrieved


In [3]:
if records and isinstance(records[0].get("credits"), dict):
    records[0]["credits"].keys()
else:
    print("No cached movie credits to inspect yet.")

In [4]:
# Table grains: each row in each table represents exactly one item or link.
# movies: one row per film.
# genres: one row per genre; movie_genres: one row per film–genre link.
# companies: one row per company; movie_companies: one row per film–company link.
# countries: one row per country; movie_origin_countries and
# movie_production_countries: one row per film–country link of that type.
# languages: one row per language; movie_spoken_languages: one row per
# film–language link.
# collections: one row per collection; people: one row per person.
# cast_credits: one row per cast member credit in one film.
# crew_credits: one row per crew member credit in one film.

from streamvault.normalize import normalize_movie_records

tables = normalize_movie_records(records)
movies = tables.movies
movies.head()


,movie_id,adult,backdrop_path,budget,homepage,imdb_id,original_language,original_title,overview,popularity,...,revenue,runtime,softcore,status,tagline,title,video,vote_average,vote_count,collection_id
0,155,False,/cfT29Im5VDvjE0RpyKOSdCKZal7.jpg,185000000,https://www.warnerbros.com/movies/dark-knight/,tt0468569,en,The Dark Knight,Batman raises the stakes in his war on crime. ...,66.8926,...,1008480529,152,False,Released,Welcome to a world without rules.,The Dark Knight,False,8.540,37405,263
1,165,False,/w6tPctxWAsHoQXeKxxNOT6lYKpx.jpg,40000000,http://www.backtothefuture.com/movies/backtoth...,tt0096874,en,Back to the Future Part II,Marty and Doc are at it again as the time-trav...,10.8613,...,332861197,108,False,Released,Getting back was only the beginning.,Back to the Future Part II,False,7.784,14528,264
2,242,False,/zNnjHLDtV8Ti3aworltaeaLMov4.jpg,54000000,NaN,tt0099674,en,The Godfather Part III,In the midst of trying to legitimize his busin...,13.7365,...,136766062,162,False,Released,All the power on earth can't change destiny.,The Godfather Part III,False,7.412,7199,230
3,49026,False,/c3OHQncTAnKFhdOTX7D3LTW6son.jpg,250000000,http://www.thedarkknightrises.com/,tt1345836,en,The Dark Knight Rises,Following the death of District Attorney Harve...,20.5660,...,1081041287,165,False,Released,The legend ends.,The Dark Knight Rises,False,7.801,25331,263
4,8077,False,/lU0zVj8KcQVkPczdGGraYGiRC72.jpg,50000000,https://www.20thcenturystudios.com/movies/alien-3,tt0103644,en,Alien³,After escaping with Newt and Hicks from the al...,7.7480,...,159773545,114,False,Released,3 times the suspense. 3 times the danger. 3 ti...,Alien³,False,6.400,6722,8091


In [5]:
table_shapes = {
    name: getattr(tables, name).shape
    for name in (
        "movies",
        "genres",
        "movie_genres",
        "companies",
        "movie_companies",
        "countries",
        "movie_origin_countries",
        "movie_production_countries",
        "languages",
        "movie_spoken_languages",
        "collections",
        "people",
        "cast_credits",
        "crew_credits",
    )
}
table_shapes

{'movies': (112, 22),
 'genres': (16, 2),
 'movie_genres': (320, 2),
 'companies': (136, 4),
 'movie_companies': (307, 2),
 'countries': (12, 2),
 'movie_origin_countries': (117, 2),
 'movie_production_countries': (154, 2),
 'languages': (27, 3),
 'movie_spoken_languages': (219, 2),
 'collections': (32, 4),
 'people': (21844, 7),
 'cast_credits': (7827, 7),
 'crew_credits': (24929, 6)}

In [6]:
from IPython.display import display

for name in (
    "genres",
    "movie_genres",
    "companies",
    "movie_companies",
    "countries",
    "movie_origin_countries",
    "movie_production_countries",
    "languages",
    "movie_spoken_languages",
    "collections",
    "people",
    "cast_credits",
    "crew_credits",
):
    frame = getattr(tables, name)
    print(f"{name}: {len(frame)} rows")
    display(frame.head())

genres: 16 rows


,genre_id,genre_name
0,28,Action
1,53,Thriller
2,80,Crime
3,12,Adventure
4,35,Comedy


movie_genres: 320 rows


,movie_id,genre_id
0,155,28
1,155,53
2,155,80
3,165,12
4,165,35


companies: 136 rows


,company_id,company_name,company_logo_path,origin_country_code
0,174,Warner Bros. Pictures,/zhD3hhtKB5qyv7ZeL4uLpNxgMVU.png,US
1,923,Legendary Pictures,/5UQsZrfbfG2dYJbx8DxfoTr2Bvu.png,US
2,9996,Syncopy,/3tvBqYsBhxWeHlu62SIJ1el93O7.png,GB
3,429,DC,/4Y00XuSMuP1gimd0jP6JT57QbCI.png,US
4,33,Universal Pictures,/8lvHyhjr8oUKOOy2dKXoALWKdp0.png,US


... (16 outputs left out)

cast_credits: 7827 rows


,movie_id,credit_id,person_id,cast_id,character,order,popularity
0,155,52fe4220c3a36847f8005d17,3894,35,Bruce Wayne,0,4.5030
1,155,52fe421fc3a36847f8005cbf,1810,3,Joker,1,3.3600
2,155,52fe4220c3a36847f8005cf1,6383,15,Harvey Dent,2,3.6508
3,155,52fe4220c3a36847f8005cf9,3895,17,Alfred,3,5.2661
4,155,52fe421fc3a36847f8005cd5,1579,8,Rachel,4,3.1646


crew_credits: 24929 rows


,movie_id,credit_id,person_id,department,job,popularity
0,155,52fe421fc3a36847f8005cdb,1213,Sound,Original Music Composer,0.8035
1,155,52fe4220c3a36847f8005d03,525,Writing,Screenplay,8.4789
2,155,52fe4220c3a36847f8005d23,10949,Production,Executive Producer,0.4825
3,155,52fe4220c3a36847f8005d29,75804,Production,Executive Producer,0.4748
4,155,52fe421fc3a36847f8005cd1,559,Camera,Director of Photography,0.7877


In [7]:
if not movies.empty and not tables.movie_genres.empty:
    before = len(tables.movie_genres)
    movie_genre_view = tables.movie_genres.merge(
        tables.genres,
        on="genre_id",
        how="left",
        validate="many_to_one",
        indicator="genre_match",
    )
    print("Genre reference coverage:")
    display(movie_genre_view["genre_match"].value_counts())
    assert len(movie_genre_view) == before, "Genre join changed the row count"

    before = len(movie_genre_view)
    movie_genre_view = movie_genre_view.merge(
        movies[["movie_id", "title"]],
        on="movie_id",
        how="left",
        validate="many_to_one",
        indicator="movie_match",
    )
    print("Movie reference coverage:")
    display(movie_genre_view["movie_match"].value_counts())
    assert len(movie_genre_view) == before, "Movie join changed the row count"
    movie_genre_view.head()
else:
    print("No movie–genre relationships to preview yet.")


Genre reference coverage:


genre_match
both          320
left_only       0
right_only      0
Name: count, dtype: int64

Movie reference coverage:


movie_match
both          320
left_only       0
right_only      0
Name: count, dtype: int64

In [8]:
source_path = project_root / "id_list.csv"
source_ids = pd.read_csv(
    source_path,
    dtype={"tmdb_id": "string"},
    keep_default_na=False,
    skip_blank_lines=False,
).rename(columns={"tmdb_id": "raw_movie_id"})
source_ids.insert(0, "source_line", range(2, len(source_ids) + 2))
source_ids["movie_id"] = pd.to_numeric(
    source_ids["raw_movie_id"], errors="coerce"
).astype("Int64")

source_coverage = source_ids.merge(
    movies[["movie_id", "title"]],
    on="movie_id",
    how="left",
    validate="many_to_one",
    indicator="cache_match",
)
assert len(source_coverage) == len(source_ids), "Cache join changed the source row count"
print("Source IDs matched to cached movies:")
display(source_coverage["cache_match"].value_counts(dropna=False))

removals_path = project_root / "id_list_removals.csv"
if removals_path.is_file():
    removals = pd.read_csv(
        removals_path, dtype="string", keep_default_na=False
    )
    removals = removals.loc[removals["source_file"].eq("id_list.csv")]
    removal_reasons = removals.rename(
        columns={
            "original_source_line": "source_line",
            "removed_value": "raw_movie_id",
            "reason": "removal_reason",
        }
    )[["source_line", "raw_movie_id", "removal_reason"]].copy()
    removal_reasons["source_line"] = pd.to_numeric(
        removal_reasons["source_line"], errors="coerce"
    ).astype("Int64")
    removal_reasons = removal_reasons.groupby(
        ["source_line", "raw_movie_id"], as_index=False
    )["removal_reason"].agg("; ".join)
else:
    removal_reasons = pd.DataFrame(
        {
            "source_line": pd.Series(dtype="Int64"),
            "raw_movie_id": pd.Series(dtype="string"),
            "removal_reason": pd.Series(dtype="string"),
        }
    )

source_coverage = source_coverage.merge(
    removal_reasons,
    on=["source_line", "raw_movie_id"],
    how="left",
    validate="one_to_one",
    indicator="removal_match",
)
assert len(source_coverage) == len(source_ids), "Removal join changed the source row count"
source_coverage["outcome"] = "No cached record; no removal reason logged"
cached = source_coverage["cache_match"].eq("both")
has_removal_reason = source_coverage["removal_match"].eq("both")
source_coverage.loc[cached, "outcome"] = "Cached"
source_coverage.loc[has_removal_reason & ~cached, "outcome"] = (
    source_coverage.loc[has_removal_reason & ~cached, "removal_reason"]
)
source_coverage.loc[cached & has_removal_reason, "outcome"] = (
    "Cached; "
    + source_coverage.loc[cached & has_removal_reason, "removal_reason"]
)
print("Source ID outcomes:")
display(source_coverage["outcome"].value_counts(dropna=False))
display(
    source_coverage[
        [
            "source_line",
            "raw_movie_id",
            "title",
            "cache_match",
            "removal_match",
            "outcome",
        ]
    ].loc[source_coverage["cache_match"].eq("left_only")]
)


Source IDs matched to cached movies:


cache_match
both          118
left_only       9
right_only      0
Name: count, dtype: int64

Source ID outcomes:


outcome
Cached                                       112
Cached; Duplicate TMDB movie ID; skipped.      6
TMDB lookup: TMDB returned HTTP 404.           5
Not a positive numeric TMDB movie ID.          3
Blank row; no TMDB movie ID.                   1
Name: count, dtype: int64

,source_line,raw_movie_id,title,cache_match,removal_match,outcome
16,18,8471293,NaN,left_only,both,TMDB lookup: TMDB returned HTTP 404.
28,30,-1,NaN,left_only,both,Not a positive numeric TMDB movie ID.
33,35,9412887,NaN,left_only,both,TMDB lookup: TMDB returned HTTP 404.
67,69,,NaN,left_only,both,Blank row; no TMDB movie ID.
71,73,8471294,NaN,left_only,both,TMDB lookup: TMDB returned HTTP 404.
91,93,9412888,NaN,left_only,both,TMDB lookup: TMDB returned HTTP 404.
93,95,tt0468569,NaN,left_only,both,Not a positive numeric TMDB movie ID.
97,99,7733901,NaN,left_only,both,TMDB lookup: TMDB returned HTTP 404.
109,111,N/A,NaN,left_only,both,Not a positive numeric TMDB movie ID.


In [9]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   movie_id           112 non-null    int64         
 1   adult              112 non-null    bool          
 2   backdrop_path      112 non-null    str           
 3   budget             112 non-null    int64         
 4   homepage           97 non-null     str           
 5   imdb_id            112 non-null    str           
 6   original_language  112 non-null    str           
 7   original_title     112 non-null    str           
 8   overview           112 non-null    str           
 9   popularity         112 non-null    float64       
 10  poster_path        112 non-null    str           
 11  release_date       112 non-null    datetime64[us]
 12  revenue            112 non-null    int64         
 13  runtime            112 non-null    int64         
 14  softcore           11

In [10]:
movies.isna().mean().mul(100).sort_values(ascending=False).rename(
    "missing_percent"
).to_frame()

,missing_percent
collection_id,14.285714
homepage,13.392857
adult,0.000000
backdrop_path,0.000000
budget,0.000000
movie_id,0.000000
imdb_id,0.000000
original_language,0.000000
overview,0.000000
original_title,0.000000


In [11]:
numeric_movies = movies.select_dtypes(include="number")
if numeric_movies.empty or numeric_movies.shape[1] == 0:
    print("No numeric movie data to summarize yet. Run `uv run streamvault` first.")
else:
    numeric_movies.describe().T